Ten skrypt czyści i tworzy od nowa schematy i tabele dla hurtowni danych

In [0]:
%sql
DROP SCHEMA IF EXISTS landing CASCADE;
CREATE SCHEMA IF NOT EXISTS landing
COMMENT 'Schemat dla danych wejściowych';

In [0]:

%sql
DROP SCHEMA IF EXISTS staging CASCADE;
CREATE SCHEMA IF NOT EXISTS staging 
COMMENT 'Schemat dla czyszczenia danych';

In [0]:
%sql
DROP SCHEMA IF EXISTS dw CASCADE;
CREATE SCHEMA IF NOT EXISTS dw
COMMENT 'Schemat dla hurtowni danych';

In [0]:

%sql
CREATE OR REPLACE TABLE staging.car (
  car_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- surrogate key
  car_inventory_id INT NOT NULL, -- business key
  car_id INT NOT NULL,
  car_producer STRING,
  car_model STRING,
  car_production_year SMALLINT,
  car_fuel_type STRING,
  car_license_plates STRING,
  car_current_rental_rate DECIMAL(18,2),
  car_purchase_price DECIMAL(18,2),
  car_sell_price DECIMAL(18,2),
  last_update TIMESTAMP,
  stag_insert_date TIMESTAMP,
  stag_update_date TIMESTAMP
) USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE dw.fact_rental (
  rental_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- surrogate key
  rental_id INT NOT NULL,  -- business key
  customer_key BIGINT NOT NULL,
  car_key BIGINT NOT NULL,
  store_key BIGINT NOT NULL,
  staff_key BIGINT NOT NULL,
  eqipment_group_key BIGINT NOT NULL,
  rental_date_key  BIGINT NOT NULL,
  return_date DATE,
  payment_date DATE,
  payment_deadline DATE,
  payment_delay_days INT NOT NULL,
  rental_amount DECIMAL(12,2) NOT NULL,
  payment_amount DECIMAL(12,2) NOT NULL,
  daily_rental_rate DECIMAL(18,2) NOT NULL,
  rental_duration INT NOT NULL
)

In [0]:
%sql
CREATE OR REPLACE TABLE dw.fact_service (
  service_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- surrogate key
  service_id INT NOT NULL,  -- business key
  car_key BIGINT NOT NULL,  -- foreign key
  service_date_key BIGINT NOT NULL, -- foreign_key
  service_type STRING NOT NULL,
  service_cost DECIMAL(18,2) NOT NULL
  ) USING DELTA;

In [0]:
%sql 
CREATE OR REPLACE TABLE dw.dim_customer ( 
  customer_key BIGINT GENERATED ALWAYS AS IDENTITY, -- surrogate key 
  customer_id INT NOT NULL, -- business key 
  customer_first_name STRING, 
  customer_last_name STRING, 
  customer_email STRING, 
  customer_birth_date DATE, 
  customer_address STRING, 
  customer_city STRING, 
  customer_country STRING, 
  customer_postal_code STRING, 
  valid_from TIMESTAMP, 
  valid_to TIMESTAMP, 
  is_active BOOLEAN, 
  dw_insert_date TIMESTAMP, 
  dw_update_date TIMESTAMP 
) USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE dw.dim_car (
  car_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- surrogate key
  car_inventory_id INT NOT NULL, -- business key
  car_id INT NOT NULL,
  car_producer STRING,
  car_model STRING,
  car_production_year SMALLINT,
  car_fuel_type STRING,
  car_license_plates STRING,
  car_current_rental_rate DECIMAL(18,2),
  car_purchase_price DECIMAL(18,2),
  car_sell_price DECIMAL(18,2),
  valid_from TIMESTAMP, -- SCD2
  valid_to TIMESTAMP, -- SCD2
  is_active BOOLEAN, -- SCD2
  dw_insert_date TIMESTAMP,
  dw_update_date TIMESTAMP
) USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE dw.dim_store ( 
  store_key BIGINT GENERATED ALWAYS AS IDENTITY, -- surrogate key 
  store_id INT NOT NULL, -- business key 
  store_manager_name STRING,
  store_manager_id INT,
  store_address STRING, 
  store_city STRING, 
  store_country STRING, 
  store_postal_code STRING, 
  valid_from TIMESTAMP, -- SCD2 
  valid_to TIMESTAMP, -- SCD2
  is_active BOOLEAN, -- SCD2  
  dw_insert_date TIMESTAMP, 
  dw_update_date TIMESTAMP
) USING DELTA; 

In [0]:
%sql 
CREATE OR REPLACE TABLE dw.dim_staff ( 
  staff_key BIGINT GENERATED ALWAYS AS IDENTITY, -- surrogate key 
  staff_id INT NOT NULL, -- business key 
  staff_first_name STRING, 
  staff_last_name STRING, 
  staff_email STRING,
  staff_address STRING,
  staff_city STRING,
  staff_country STRING,
  staff_postal_code STRING,
  staff_store_id INT, 
  staff_hired_date DATE, 
  staff_manager_id INT,
  valid_from TIMESTAMP, -- SCD2 
  valid_to TIMESTAMP, -- SCD2 
  is_active BOOLEAN, -- SCD2 
  dw_insert_date TIMESTAMP, 
  dw_update_date TIMESTAMP 
) USING DELTA; 

In [0]:
%sql
CREATE OR REPLACE TABLE dw.dim_rental_date (
  rental_date_key INT NOT NULL, -- surrogate key
  rental_date DATE NOT NULL, -- business key
  rental_calendar_year SMALLINT NOT NULL,
  rental_calendar_quarter TINYINT NOT NULL,
  rental_calendar_month TINYINT NOT NULL,
  rental_calendar_day TINYINT NOT NULL,
  rental_days_since_epoch INT NOT NULL,
  rental_day_of_week TINYINT NOT NULL,
  rental_week_of_year TINYINT NOT NULL,
  rental_is_working_day BOOLEAN NOT NULL,
  rental_is_weekend BOOLEAN NOT NULL,
  rental_is_covid BOOLEAN NOT NULL
) USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE dw.dim_service_date (
  service_date_key INT NOT NULL, -- surrogate key
  service_date DATE NOT NULL, -- business key
  service_calendar_year SMALLINT NOT NULL,
  service_calendar_quarter TINYINT NOT NULL,
  service_calendar_month TINYINT NOT NULL,
  service_calendar_day TINYINT NOT NULL,
  service_days_since_epoch INT NOT NULL,
  service_day_of_week TINYINT NOT NULL,
  service_week_of_year TINYINT NOT NULL
) USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE dw.bridge_staff_hierarchy ( 
  employee_id INT, 
  manager_id INT, 
  distance TINYINT, 
  dw_insert_date TIMESTAMP 
) USING DELTA; 

In [0]:
%sql
CREATE OR REPLACE TABLE dw.dim_equipment ( 
  equipment_key BIGINT GENERATED ALWAYS AS IDENTITY, -- surrogate key 
  equipment_id INT NOT NULL, -- business key 
  equipment_name STRING, 
  equipment_type STRING, 
  equipment_version STRING,
  dw_insert_date TIMESTAMP, 
  dw_update_date TIMESTAMP 
) USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE dw.dim_equipment_group (
  equipment_group_key BIGINT GENERATED ALWAYS AS IDENTITY, -- surrogate key 
  equipment_group_id INT NOT NULL,
  equipment_group_name STRING NOT NULL, 
  is_active BOOLEAN NOT NULL,
  dw_insert_date TIMESTAMP, 
  dw_update_date TIMESTAMP 
)

In [0]:
%sql
CREATE OR REPLACE TABLE dw.bridge_equipment_group_equipment (
  equipment_group_key BIGINT,
  equipment_key BIGINT,
  dw_insert_date TIMESTAMP
)

In [0]:
%sql 
CREATE OR REPLACE TABLE dw.calendar AS 
WITH calendarDates AS ( 
  SELECT explode( 
    sequence( 
      make_date(2016, 01, 01), 
      make_date(2050, 01, 01), 
      interval 1 day 
    ) 
  ) AS calendar_date 
) 
SELECT 
  10000 * year(calendar_date) + 100 * month(calendar_date) + day(calendar_date) AS calendar_date_key, 
  to_date(calendar_date) AS calendar_date, 
  year(calendar_date) AS calendar_year, 
  quarter(calendar_date) AS calendar_quarter, 
  month(calendar_date) AS calendar_month, 
  day(calendar_date) AS calendar_day, 
  dayofweek(calendar_date) AS day_of_week, 
  weekofyear(calendar_date) AS calendar_iso_week_of_year,
  CASE  
    WHEN dayofweek(calendar_date) IN (2,3,4,5,6) THEN true  
    ELSE false  
  END  AS is_working_day,  
  CASE  
    WHEN dayofweek(calendar_date) IN (1, 7) THEN true  
    ELSE false  
  END  AS is_weekend,     
  date_format(calendar_date, 'EEEE') AS calendar_day_name, 
  datediff(calendar_date,'2000-01-01') AS days_since_epoch, 
  CASE  
    WHEN calendar_date BETWEEN date '2020-03-01' AND date '2022-02-28' 
      THEN true 
    ELSE false 
  END AS is_covid 
FROM calendarDates; 

In [0]:
%sql 
CREATE OR REPLACE TABLE dw.dim_manager ( 
  manager_key BIGINT GENERATED ALWAYS AS IDENTITY, -- surrogate key 
  manager_id INT NOT NULL, -- business key 
  manager_first_name STRING, 
  manager_last_name STRING, 
  manager_email STRING, 
  manager_address STRING,
  manager_city STRING,
  manager_country STRING,
  manager_postal_code STRING,
  manager_store_id INT,
  manager_hired_date DATE, 
  valid_from TIMESTAMP, -- SCD2 
  valid_to TIMESTAMP, -- SCD2 
  is_active BOOLEAN, -- SCD2 
  dw_insert_date TIMESTAMP, 
  dw_update_date TIMESTAMP 
) USING DELTA; 

In [0]:
%sql
INSERT INTO dw.dim_rental_date (
  rental_date_key,
  rental_date,
  rental_calendar_year,
  rental_calendar_quarter,
  rental_calendar_month,
  rental_calendar_day,
  rental_days_since_epoch,
  rental_day_of_week,
  rental_week_of_year,
  rental_is_working_day,
  rental_is_weekend,
  rental_is_covid
)
SELECT
  calendar_date_key,
  calendar_date,
  calendar_year,
  calendar_quarter,
  calendar_month,
  calendar_day,
  days_since_epoch,
  day_of_week,
  calendar_iso_week_of_year,
  is_working_day,
  is_weekend,
  is_covid
FROM dw.calendar;

In [0]:
%sql
INSERT INTO dw.dim_service_date (
  service_date_key,
  service_date,
  service_calendar_year,
  service_calendar_quarter,
  service_calendar_month,
  service_calendar_day,
  service_days_since_epoch,
  service_day_of_week,
  service_week_of_year
)
SELECT
  calendar_date_key,
  calendar_date,
  calendar_year,
  calendar_quarter,
  calendar_month,
  calendar_day,
  days_since_epoch,
  day_of_week,
  calendar_iso_week_of_year
FROM dw.calendar;